In [ ]:
import os
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv

load_dotenv()

# Set Tavily API Key
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

# 1. Initialize Tavily Search Tool
search_tool = TavilySearchResults(max_results=2)

def human_assistance(query:str):
    """Use this tool to ask the user for assistance"""
    human_responese= interrupt({"query":query})
    return human_responese["data"]

# 2. Bind Tools to Gemini Model
llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash")
tools=[human_assistance,search_tool]
llm_with_tools = llm.bind_tools(tools)

# 3. Define Graph State
class State(TypedDict):
    messages: Annotated[list, add_messages]

# 4. Define Nodes
def chatbot(state: State):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

tool_node = ToolNode(tools=tools)

# 5. Build Graph with Conditional Routing
builder = StateGraph(State)
builder.add_node("chatbot", chatbot)
builder.add_node("tools", tool_node)

builder.add_edge(START, "chatbot")
# Automatically routes to "tools" if LLM generates a tool call, otherwise to END
builder.add_conditional_edges("chatbot", tools_condition)
builder.add_edge("tools", "chatbot")

graph = builder.compile()


In [8]:
user_input = "I need some expert guidance for building an AI agent. Could you request assisstance for me?"
config = {"configurable": {"thread_id": "123"}}

events= graph.stream({"messages": user_input}, config=config,stream_mode="values")
for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()

================================ Human Message =================================

I need some expert guidance for building an AI agent. Could you request assisstance for me?


RateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}

In [ ]:
human_message =("LangGraph is a framework for building and running multi-step workflows with LLMs")
human_command = Command(resume={"data":human_message})

events= graph.stream(human_command, config=config,stream_mode="values")
for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()

RuntimeError: Cannot use Command(resume=...) without checkpointer